<a href="https://colab.research.google.com/github/ricardoalmeida/ai-post-generator/blob/main/Google_ADK_AI_Agent_for_social_media.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-genai google-adk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.1/232.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.1/217.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.1/334.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.8/65.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.0/119.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.9/194.9 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.

In [3]:
# Configura a API Key do Google Gemini

import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [4]:
# Configura o cliente da SDK do Gemini

from google import genai

client = genai.Client()

In [5]:
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search
from google.genai import types  # Para criar conteúdos (Content e Part)

In [6]:
# Cria um serviço de sessão em memória

session_service = InMemorySessionService()

In [7]:
# Função auxiliar que envia uma mensagem para um agente via Runner e retorna a resposta final

async def call_agent(agent: Agent, message_text: str) -> str:
    # Cria uma nova sessão (você pode personalizar os IDs conforme necessário)
    session = await session_service.create_session(app_name=agent.name, user_id="user1")
    # Cria um Runner para o agente
    runner = Runner(agent=agent, app_name=agent.name, session_service=session_service)
    # Cria o conteúdo da mensagem de entrada
    content = types.Content(role="user", parts=[types.Part(text=message_text)])

    final_response = ""
    # Itera assincronamente pelos eventos retornados durante a execução do agente
    async for event in runner.run_async(user_id="user1", session_id=session.id, new_message=content):
        if event.is_final_response():
          for part in event.content.parts:
            if part.text is not None:
              final_response += part.text
              final_response += "\n"
    return final_response

In [8]:
# Lista todos os modelos disponíveis atualmente

for model in client.models.list():
    print(model.name)

models/embedding-gecko-001
models/gemini-1.0-pro-vision-latest
models/gemini-pro-vision
models/gemini-1.5-pro-latest
models/gemini-1.5-pro-001
models/gemini-1.5-pro-002
models/gemini-1.5-pro
models/gemini-1.5-flash-latest
models/gemini-1.5-flash-001
models/gemini-1.5-flash-001-tuning
models/gemini-1.5-flash
models/gemini-1.5-flash-002
models/gemini-1.5-flash-8b
models/gemini-1.5-flash-8b-001
models/gemini-1.5-flash-8b-latest
models/gemini-1.5-flash-8b-exp-0827
models/gemini-1.5-flash-8b-exp-0924
models/gemini-2.5-pro-exp-03-25
models/gemini-2.5-pro-preview-03-25
models/gemini-2.5-flash-preview-04-17
models/gemini-2.5-flash-preview-05-20
models/gemini-2.5-flash-preview-04-17-thinking
models/gemini-2.5-pro-preview-05-06
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-preview-image-generation
models/gemini-2.0-flash-lite-preview

In [9]:
MODELO_RAPIDO = "gemini-2.0-flash"
MODELO_ROBUSTO = "gemini-2.5-pro-preview-03-25"

In [10]:
##########################################
# --- Agente 1: Buscador de Notícias --- #
##########################################

async def agente_buscador(topico, data_de_hoje):

    buscador = Agent(
        name="agente_buscador",
        model=MODELO_RAPIDO,
        instruction="""
        Você é um assistente de pesquisa. A sua tarefa é usar a ferramenta de busca do google (google_search)
        para recuperar as últimas notícias de lançamentos muito relevantes sobre o tópico abaixo.
        Foque em no máximo 5 lançamentos relevantes, com base na quantidade e entusiasmo das notícias sobre ele.
        Se um tema tiver poucas notícias ou reações entusiasmadas, é possível que ele não seja tão relevante assim
        e pode ser substituído por outro que tenha mais.
        Esses lançamentos relevantes devem ser atuais, de no máximo um mês antes da data de hoje.
        """,
        description="Agente que busca informações no Google",
        tools=[google_search]
    )

    entrada_do_agente_buscador = f"Tópico: {topico}\nData de hoje: {data_de_hoje}"

    # Executa o agente
    lancamentos = await call_agent(buscador, entrada_do_agente_buscador)
    return lancamentos

In [11]:
################################################
# --- Agente 2: Planejador de posts --- #
################################################

async def agente_planejador(topico, lancamentos_buscados):
    planejador = Agent(
        name="agente_planejador",
        model=MODELO_RAPIDO,
        instruction="""
        Você é um planejador de conteúdo, especialista em redes sociais.
        Você recebe uma lista de lançamentos recentes do agente buscador e,
        - Para cada um dos lançamentos recebidos, você deve usar a ferramenta
        de busca do Google (google_search) para buscar os pontos mais relevantes
        que poderíamos abordar em um post sobre cada um deles.
        Você também pode usar o (google_search) para encontrar mais
        informações sobre cada um dos temas e aprofundar.
        - Depois de terminar a busca, você irá escolher APENAS UM tema dentre todos eles,
        aquele tema com potencial de ser o mais relevante com base nas suas pesquisas.
        - Depois de escolher o tema mais relevante dentre todos eles, você irá retornar
        qual foi o tema escolhido, seus pontos mais relevantes, e um plano com os assuntos
        a serem abordados no post que será escrito posteriormente.
        """,
        description="Agente que planeja posts",
        tools=[google_search]
    )

    entrada_do_agente_planejador = f"Tópico:{topico}\nLançamentos buscados: {lancamentos_buscados}"

    # Executa o agente
    plano_do_post = await call_agent(planejador, entrada_do_agente_planejador)
    return plano_do_post

In [16]:
######################################
# --- Agente 3: Redator do Post --- #
######################################

async def agente_redator(topico, plano_de_post):
    redator = Agent(
        name="agente_redator",
        model=MODELO_RAPIDO,
        instruction="""
            Você é um Redator Criativo especializado em criar posts virais para redes sociais.
            Você escreve posts para a empresa Alura, a maior escola online de tecnologia do Brasil.
            Utilize o tema fornecido no plano de post e os pontos mais relevantes fornecidos e, com base nisso,
            escreva um post para Instagram sobre o tema indicado.
            O post deve ser engajador, informativo, com linguagem simples e incluir 2 a 4 hashtags no final.
            """,
        description="Agente redator de posts engajadores para Instagram"
    )
    entrada_do_agente_redator = f"Tópico: {topico}\nPlano de post: {plano_de_post}"

    # Executa o agente
    post_final = await call_agent(redator, entrada_do_agente_redator)
    return post_final

In [13]:
import textwrap # Para formatar melhor a saída de texto
from IPython.display import display, Markdown # Para exibir texto formatado no Colab

# Função auxiliar para exibir texto formatado em Markdown no Colab
def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

In [14]:
from datetime import date

data_de_hoje = date.today().strftime("%d/%m/%Y")

In [17]:
print("🚀 Iniciando o Sistema de Criação de Posts para Instagram com 3 Agentes 🚀\n")

# --- Obter o Tópico do Usuário ---
topico = input("❓ Por favor, digite o TÓPICO sobre o qual você quer criar o post de tendências: ")

# Inserir lógica do sistema de agentes
if not topico:
    print("\nVocê esqueceu de digitar o tópico!")
else:
    print(f"\nMaravilha! Vamos então criar o post sobre novidades em {topico}")

    lancamentos_buscados = await agente_buscador(topico, data_de_hoje)
    print("\n--- 📝 Resultado do Agente 1 (Buscador) ---\n")
    display(to_markdown(lancamentos_buscados))
    print("--------------------------------------------------------------")

    plano_de_post = await agente_planejador(topico, lancamentos_buscados)
    print("\n--- 📝 Resultado do Agente 2 (Planejador) ---\n")
    display(to_markdown(plano_de_post))
    print("--------------------------------------------------------------")

    post_redigido = await agente_redator(topico, plano_de_post)
    print("\n--- 📝 Resultado do Agente 3 (Redator) ---\n")
    display(to_markdown(post_redigido))
    print("--------------------------------------------------------------")

🚀 Iniciando o Sistema de Criação de Posts para Instagram com 3 Agentes 🚀

❓ Por favor, digite o TÓPICO sobre o qual você quer criar o post de tendências: Tendências em Inteligência Artificial

Maravilha! Vamos então criar o post sobre novidades em Tendências em Inteligência Artificial

--- 📝 Resultado do Agente 1 (Buscador) ---



> Ok, aqui estão as principais tendências recentes em Inteligência Artificial que encontrei, focando em lançamentos e desenvolvimentos notáveis de aproximadamente o último mês:
> 
> 
> Com base nas minhas pesquisas, aqui estão algumas das tendências de IA mais relevantes e lançamentos recentes (último mês - Abril/Maio 2025):
> 
> 1.  **Agentes de IA e Autonomia:** Agentes de IA mais inteligentes, capazes de automatizar tarefas complexas e otimizar processos em vários setores. Há uma mudança para que esses agentes se tornem "membros" das equipes, realizando tarefas operacionais e liberando os humanos para atividades mais estratégicas. (Fontes: 1, 14, 15)
> 2.  **Acessibilidade e Democratização da IA:** A IA está se tornando mais acessível devido aos custos de treinamento mais baixos e à disponibilidade de ferramentas no-code e low-code, permitindo que mais empresas implementem soluções de IA personalizadas. (Fontes: 5, 14)
> 3.  **IA Generativa Avançada:** Ferramentas como o Gemini da Google estão evoluindo, com lançamentos como o Gemini 2.5 Pro e Flash, oferecendo desempenho superior e custos mais baixos. Há também avanços em geração de conteúdo, como Imagen 4 e Veo 3, que revolucionam a criação de vídeos e imagens. (Fonte: 4)
> 4.  **IA Explicável (XAI) e Ética:** A transparência e a explicabilidade dos sistemas de IA estão se tornando prioridades, com foco em garantir que os humanos possam entender os processos de tomada de decisão da IA, especialmente em áreas críticas como medicina e finanças. (Fontes: 11, 8)
> 5.  **IA na Otimização de Processos e Automação Inteligente:** A IA está sendo amplamente adotada para otimizar operações, reduzir custos e atender às demandas dos clientes em vários setores, incluindo cadeia de suprimentos, logística e compras. Ela facilita a previsão de demanda, otimização de estoque e automatização de tarefas. (Fontes: 2, 3)
> 6. **Edge AI:** A computação de IA está sendo movida para mais perto de onde os dados são gerados e consumidos com o Edge AI da Azion. Isso permite que as empresas implementem IA, reduzindo custos operacionais, acelerando insights e mantendo o controle sobre dados confidenciais. (Fonte: 12)
> 


--------------------------------------------------------------

--- 📝 Resultado do Agente 2 (Planejador) ---



> Ok, com base nos lançamentos e tendências recentes em Inteligência Artificial que você me forneceu, vou usar a ferramenta de busca do Google para aprofundar meu conhecimento sobre cada um deles e identificar o tema mais relevante para um post de mídia social.
> 
> 
> Após uma análise mais aprofundada das informações coletadas sobre as tendências de IA, acredito que o tema **"Agentes de IA e Autonomia"** possui o maior potencial para gerar engajamento e discussões relevantes nas redes sociais.
> 
> **Pontos Relevantes:**
> 
> *   **Capacidade de automatizar tarefas complexas:** Os agentes de IA estão evoluindo para realizar tarefas operacionais em diversos setores, otimizando processos e liberando equipes humanas para atividades mais estratégicas.
> *   **Integração nas equipes de trabalho:** A perspectiva de agentes de IA se tornarem "membros" das equipes, auxiliando na execução de tarefas e otimização de processos, é um tema inovador e com grande potencial de discussão.
> *   **Aplicações práticas e casos de uso:** Há exemplos concretos de como os agentes de IA estão sendo implementados em empresas de diversos setores, gerando resultados positivos e transformando a forma como o trabalho é realizado.
> 
> **Plano de Assuntos para o Post:**
> 
> 1.  **Título Atraente:** Algo como "Agentes de IA: O Futuro do Trabalho Já Começou?" ou "IA Autônoma: Prepare-se para Trabalhar com Robôs Inteligentes".
> 2.  **Introdução:** Apresentar o conceito de agentes de IA, destacando sua capacidade de automatizar tarefas complexas e otimizar processos. Mencionar a previsão de que agentes de IA "entrarão no mercado de trabalho" em breve.
> 3.  **O que são Agentes de IA?** Explicar o que são agentes de IA e como eles funcionam, mencionando tecnologias como processamento de linguagem natural (PNL) e aprendizado de máquina (ML).
> 4.  **Benefícios e Vantagens:** Abordar os benefícios da utilização de agentes de IA, como aumento da eficiência, redução de custos, otimização de processos e liberação de equipes humanas para atividades mais estratégicas.
> 5.  **Exemplos e Casos de Uso:** Apresentar exemplos de empresas que já estão utilizando agentes de IA em seus processos, mencionando os resultados positivos obtidos. Citar a Anthropic e outras empresas que estão desenvolvendo agentes avançados.
> 6.  **O Futuro dos Agentes de IA:** Discutir o futuro dos agentes de IA, mencionando a evolução da tecnologia, a capacidade de usar ferramentas de forma mais eficiente e a integração em diversos setores.
> 7.  **Considerações Éticas:** Abordar as considerações éticas relacionadas à utilização de agentes de IA, como a necessidade de transparência, responsabilidade e garantia de que a tecnologia seja utilizada de forma justa e imparcial.
> 8.  **Chamada para Ação:** Incentivar os seguidores a compartilhar suas opiniões sobre o tema, perguntando se eles estão preparados para trabalhar com agentes de IA e como imaginam o futuro do trabalho com a tecnologia.
> 
> Este plano permitirá criar um post informativo, envolvente e com potencial para gerar discussões relevantes sobre o futuro do trabalho e o papel da IA na transformação das empresas.
> 


--------------------------------------------------------------

--- 📝 Resultado do Agente 3 (Redator) ---



> 🚀🤖 **Agentes de IA: O Futuro do Trabalho Já Começou?** 🤖🚀
> 
> Já imaginou ter um colega de trabalho que nunca se cansa, aprende rapidinho e ainda te ajuda a otimizar todas as tarefas? 🤔 Bem-vindos à era dos Agentes de IA!
> 
> ✨ **O que são?** ✨
> Agentes de IA são sistemas inteligentes que usam PNL e Machine Learning para automatizar tarefas complexas e otimizar processos. Eles estão chegando para revolucionar a forma como trabalhamos!
> 
> 📈 **Benefícios? Temos!** 📈
> ✅ Aumento da eficiência
> ✅ Redução de custos
> ✅ Otimização de processos
> ✅ Equipes humanas focadas em tarefas estratégicas
> 
> 🌐 **Quem já está usando?** 🌐
> Empresas de diversos setores já estão implementando Agentes de IA e colhendo resultados incríveis! A Anthropic e outras gigantes estão na vanguarda dessa transformação.
> 
> 🔮 **O Futuro? É agora!** 🔮
> Prepare-se para um futuro onde a IA te ajuda a usar as ferramentas de forma mais eficiente e te impulsiona em todas as áreas!
> 
> 🤔 **E a ética?** 🤔
> Claro, a gente não esquece! Transparência, responsabilidade e uso justo são pilares essenciais nessa jornada.
> 
> 💬 **E aí, preparado(a) para trabalhar lado a lado com robôs inteligentes? Compartilhe sua opinião! Como você imagina o futuro do trabalho com a IA?** 👇
> 
> #InteligenciaArtificial #IA #FuturoDoTrabalho #Alura
> 


--------------------------------------------------------------
